# Math 280: Mathematical and Statistical Foundations of Data Science
## Week 13 — Compression and Reconstruction: PCA, NMF, and Autoencoders
### UC Merced | Spring 2026 | S. Sindi

---

This notebook accompanies the Week 13 Lecture 1 notes. The unifying theme is **reconstruction**: all three methods we study try to compress data into a low-dimensional representation and then reconstruct the original from that representation. They differ only in what constraints they impose.

The notebook is in two parts:

| Part | Method | Data | What you will see |
|------|--------|------|-------------------|
| 1 | PCA (truncated SVD) | Single image (BTTF.tiff) | Image reconstruction at increasing rank $k$ |
| 2 | PCA vs NMF vs Autoencoder | Olivetti faces dataset | How each method's *components* differ |

**Run every cell in order. Read the text between cells carefully — it explains what you are seeing and why it matters mathematically.**

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from PIL import Image                          # for loading the TIFF
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA, NMF
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Path to the image — adjust if your local layout differs
IMG_PATH = Path('../../Data/BTTF.tiff')

print('Ready.')

---
## Part 1 — PCA as Image Compression

### The Eckart–Young Theorem in Action

Recall from the lecture notes: the **rank-$k$ truncated SVD** is the best possible rank-$k$ approximation to a matrix in the Frobenius norm. This is the Eckart–Young theorem.

An image is just a matrix (or three matrices, one per colour channel). If we compute the SVD of each channel and keep only the top $k$ singular triplets, we get the best rank-$k$ approximation to that channel — which is also the best rank-$k$ reconstruction we could possibly achieve with *any* method.

The question is: **how large does $k$ need to be before the image looks good?** The answer reveals how much redundancy (low-dimensional structure) the image contains.

In [ ]:
# ── Load and inspect the image ──────────────────────────────────────────────
img_pil  = Image.open(IMG_PATH).convert('RGB')   # force RGB (drop alpha if present)
img      = np.array(img_pil, dtype=float) / 255.0  # scale pixels to [0, 1]

H, W, C  = img.shape
print(f'Image size : {H} rows × {W} columns × {C} channels')
print(f'Each channel is a {H}×{W} matrix.')
print(f'Max possible rank : min({H}, {W}) = {min(H, W)}')

# Quick look at the original
fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(img)
ax.set_title('Original image — Doc and Marty, BTTF')
ax.axis('off')
plt.tight_layout()
plt.show()

### The SVD of Each Colour Channel

We compute the SVD separately for each of the R, G, and B channels. Each channel gives us a set of singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r \geq 0$.

The **scree plot** shows these singular values. An elbow in the curve indicates where additional components contribute diminishing returns — exactly the same diagnostic we use for choosing $k$ in PCA on tabular data.

In [ ]:
# ── Compute the full SVD for each channel ───────────────────────────────────
# numpy's linalg.svd returns U (H×H), s (singular values), Vt (W×W)
# We use full_matrices=False to get the thin SVD — more efficient
svds = []
for c in range(C):
    U, s, Vt = np.linalg.svd(img[:, :, c], full_matrices=False)
    svds.append((U, s, Vt))

channel_names = ['Red', 'Green', 'Blue']
colours       = ['#e04a4a', '#4a9e4a', '#4a6ee0']

# ── Scree plot: singular values for each channel ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

n_show = 80   # show the first 80 singular values — the rest are tiny

for c, (U, s, Vt) in enumerate(svds):
    axes[0].plot(s[:n_show], color=colours[c], label=channel_names[c], linewidth=1.8)
    # Cumulative variance explained = cumulative sum of s^2 / total sum of s^2
    var_explained = np.cumsum(s**2) / np.sum(s**2)
    axes[1].plot(var_explained[:n_show] * 100, color=colours[c],
                 label=channel_names[c], linewidth=1.8)

axes[0].set_xlabel('Singular value index $i$')
axes[0].set_ylabel('$\\sigma_i$')
axes[0].set_title('Scree plot — singular values')
axes[0].legend()

axes[1].set_xlabel('Rank $k$')
axes[1].set_ylabel('Cumulative variance explained (%)')
axes[1].set_title('Variance explained by top-$k$ components')
# Draw reference lines at 80%, 90%, 95%
for pct in [80, 90, 95]:
    axes[1].axhline(pct, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    axes[1].text(n_show * 0.98, pct + 0.5, f'{pct}%', ha='right',
                 fontsize=8, color='grey')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print how many components needed for 90% and 95% variance in the green channel
s_g = svds[1][1]   # green channel singular values
var_g = np.cumsum(s_g**2) / np.sum(s_g**2)
k90 = np.searchsorted(var_g, 0.90) + 1
k95 = np.searchsorted(var_g, 0.95) + 1
print(f'Green channel: {k90} components explain 90% of variance')
print(f'Green channel: {k95} components explain 95% of variance')
print(f'Full rank of the image: {min(H, W)}')
print(f'Compression ratio at k={k90}: {min(H,W)/k90:.0f}x')

### Rank-$k$ Reconstruction

The rank-$k$ approximation of channel matrix $X$ is:
$$\hat{X}_k = U_k \Sigma_k V_k^\top$$
where $U_k, \Sigma_k, V_k$ keep only the top $k$ singular triplets.

By the Eckart–Young theorem, no rank-$k$ matrix does better than this in Frobenius norm. The reconstruction error is exactly $\sum_{j > k} \sigma_j^2$.

We reconstruct the image at several values of $k$ and display them side by side. Watch how quickly the image becomes recognisable — this tells you how much low-dimensional structure the image contains.

In [ ]:
def reconstruct_image(svds, k):
    """
    Reconstruct the RGB image using only the top-k singular triplets.
    Each channel is reconstructed independently: X_hat = U_k @ diag(s_k) @ Vt_k
    Pixel values are clipped to [0, 1] after reconstruction.
    """
    channels = []
    for U, s, Vt in svds:
        # Keep only the top-k components
        X_hat = (U[:, :k] * s[:k]) @ Vt[:k, :]   # equivalent to U_k @ diag(s_k) @ Vt_k
        X_hat = np.clip(X_hat, 0, 1)              # clip to valid pixel range
        channels.append(X_hat)
    return np.stack(channels, axis=2)             # reassemble (H, W, 3)


def frobenius_error(svds, k):
    """
    Relative Frobenius reconstruction error averaged across channels.
    = sum_{j > k} sigma_j^2  /  sum_j sigma_j^2
    """
    errors = []
    for U, s, Vt in svds:
        err = np.sum(s[k:]**2) / np.sum(s**2)
        errors.append(err)
    return np.mean(errors)


# ── Show reconstructions at increasing rank ──────────────────────────────────
k_values = [1, 5, 10, 20, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, k in zip(axes, k_values):
    recon = reconstruct_image(svds, k)
    err   = frobenius_error(svds, k)
    ax.imshow(recon)
    ax.set_title(f'Rank $k = {k}$\nerror = {err:.3f}', fontsize=10)
    ax.axis('off')

fig.suptitle(
    'Rank-$k$ SVD reconstruction — the Eckart–Young theorem in action\n'
    '(error = relative Frobenius reconstruction error, averaged across R/G/B)',
    fontsize=12
)
plt.tight_layout()
plt.show()

### What Are the Singular Vectors?

It is worth pausing to ask: what do the individual components look like?

The left singular vectors $u_j$ (columns of $U$) are spatial patterns in the row direction. The right singular vectors $v_j$ (rows of $V^\top$) are spatial patterns in the column direction. Their outer product $\sigma_j u_j v_j^\top$ is the $j$-th rank-1 "layer" of the image.

The first few layers capture the coarsest, highest-energy structure; later layers capture finer detail. Together they reconstruct the image, but each individual layer has a ghostly, wave-like appearance — **not** a meaningful part of the image. This is the key contrast with NMF, which we see in Part 2.

In [ ]:
# ── Visualise individual rank-1 layers for the green channel ─────────────────
U_g, s_g, Vt_g = svds[1]   # green channel

layers_to_show = [0, 1, 2, 3, 4, 9, 19, 49]   # 0-indexed

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, j in zip(axes, layers_to_show):
    # Rank-1 layer: sigma_j * u_j * v_j^T
    layer = s_g[j] * np.outer(U_g[:, j], Vt_g[j, :])
    # Normalise to [0,1] for display (layers can have negative values)
    layer_norm = (layer - layer.min()) / (layer.max() - layer.min() + 1e-8)
    ax.imshow(layer_norm, cmap='gray')
    ax.set_title(f'Layer $j = {j+1}$\n$\\sigma_{{{j+1}}} = {s_g[j]:.1f}$', fontsize=9)
    ax.axis('off')

fig.suptitle(
    'Individual rank-1 SVD layers (green channel)\n'
    r'Each layer = $\sigma_j \, u_j v_j^\top$ — global, wave-like, not interpretable as "parts"',
    fontsize=11
)
plt.tight_layout()
plt.show()

---
## Part 2 — PCA vs NMF vs Autoencoder on Faces

The single-image demo illustrates reconstruction quality. But to see **why NMF and autoencoders exist**, we need a dataset of many images — so the methods can learn what structure is shared across images.

We use the **Olivetti faces dataset**: 400 grayscale face photographs of 40 people, 10 photos each, at 64×64 pixels. Each face is a vector in $\mathbb{R}^{4096}$.

In this case, they are operating on the combined data:
$\mathbb{R}^{400 \times 4096}$.

The three methods will each find a set of $k = 16$ **components** — building blocks from which any face can be approximately reconstructed. The components themselves are the story.

- **PCA components** (eigenfaces): global, wave-like, contain positive and negative values — hard to interpret as parts of a face.
- **NMF components**: non-negative, localised — look like eyes, noses, foreheads. These are the "parts" that Lee & Seung described in their 1999 *Nature* paper.
- **Autoencoder components**: the decoder weight vectors — nonlinear, learned representations that don't have a clean visual interpretation but produce the best reconstructions.

In [ ]:
# ── Load the Olivetti faces dataset ──────────────────────────────────────────
# 400 images, each flattened to a 4096-dimensional vector (64x64 pixels)
faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
X     = faces.data          # shape: (400, 4096), values already in [0, 1]

print(f'Dataset shape : {X.shape}')
print(f'  {X.shape[0]} images, each {X.shape[1]} pixels (= 64 × 64)')
print(f'Pixel range   : [{X.min():.3f}, {X.max():.3f}]')

# ── Show a sample of faces ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.flatten(), range(16)):
    ax.imshow(X[i].reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
fig.suptitle('A sample of Olivetti faces (400 images, 64×64 pixels each)', fontsize=11)
plt.tight_layout()
plt.show()

### Method 1: PCA — Eigenfaces

PCA finds the $k$ directions of maximum variance in the data. Each direction is a vector in $\mathbb{R}^{4096}$ — when reshaped to 64×64, it is a *ghostly face-like image* called an **eigenface**.

Because PCA imposes orthogonality but no sign constraint, the components can have both positive and negative pixel values. They look like superpositions of facial features with positive and negative weights — mathematically optimal, but not decomposable into meaningful parts.

In [ ]:
K = 16   # number of components for all three methods — same k for fair comparison

# ── Fit PCA ──────────────────────────────────────────────────────────────────
pca = PCA(n_components=K, random_state=SEED)
pca.fit(X)

# pca.components_ has shape (K, 4096) — each row is a principal direction
# pca.explained_variance_ratio_ gives the fraction of variance each component explains

print(f'Variance explained by {K} PCA components: '
      f'{pca.explained_variance_ratio_.sum():.1%}')

# ── Plot the eigenfaces ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, comp in zip(axes.flatten(), pca.components_):
    face = comp.reshape(64, 64)
    # Use a diverging colourmap to show positive (bright) and negative (dark) regions
    ax.imshow(face, cmap='RdBu_r',
              vmin=-np.abs(face).max(), vmax=np.abs(face).max())
    ax.axis('off')
fig.suptitle(
    f'PCA components (eigenfaces) — $k = {K}$\n'
    'Red = positive weight, Blue = negative weight.\n'
    'Global, wave-like structure — not interpretable as face parts.',
    fontsize=10
)
plt.tight_layout()
plt.show()

### Method 2: NMF — Parts of Faces

NMF solves the same reconstruction problem as PCA — minimise $\|X - WH\|_F^2$ — but adds the constraint that both $W$ and $H$ must be **non-negative**.

Because pixels are already non-negative (intensity values), this constraint forces the components to be additive: each face is a non-negative sum of non-negative parts. There is no cancellation between positive and negative regions.

The result, as Lee & Seung showed in 1999, is components that look like localised **parts of faces** — eyes, nose, forehead, cheek — rather than global wave patterns.

In [ ]:
# ── Fit NMF ──────────────────────────────────────────────────────────────────
# NMF requires non-negative input. The Olivetti data is already in [0,1], so we're fine.
# init='nndsvda' is a smarter initialisation than random — faster convergence.
nmf = NMF(n_components=K, init='nndsvda', random_state=SEED, max_iter=1000, tol=1e-4)
nmf.fit(X)

# nmf.components_ has shape (K, 4096) — each row is a non-negative basis component
# Unlike PCA, NMF has no built-in variance explained — we compute reconstruction error
X_nmf_recon = nmf.transform(X) @ nmf.components_   # W @ H
nmf_error   = np.mean((X - X_nmf_recon)**2)
print(f'NMF mean squared reconstruction error : {nmf_error:.5f}')

# Also compute PCA reconstruction error for comparison
X_pca_recon = pca.inverse_transform(pca.transform(X))
pca_error   = np.mean((X - X_pca_recon)**2)
print(f'PCA mean squared reconstruction error : {pca_error:.5f}')
print(f'(PCA has lower error — it is the optimal linear method by Eckart–Young)')

# ── Plot the NMF components ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, comp in zip(axes.flatten(), nmf.components_):
    face = comp.reshape(64, 64)
    ax.imshow(face, cmap='gray', vmin=0, vmax=face.max())
    ax.axis('off')
fig.suptitle(
    f'NMF components — $k = {K}$\n'
    'All values are non-negative — no cancellation.\n'
    'Components localise to face parts: eyes, nose, forehead, cheeks.',
    fontsize=10
)
plt.tight_layout()
plt.show()

### Side-by-Side Reconstruction Comparison: PCA vs NMF

Both methods reconstruct faces from $k = 16$ components. PCA will generally have lower reconstruction error (it is provably optimal). But the question is not just *how well* they reconstruct — it is *what structure they find* in doing so.

In [ ]:
# ── Reconstruct a handful of faces with PCA and NMF ──────────────────────────
n_faces  = 6
face_ids = [0, 10, 20, 30, 40, 50]   # pick some faces spread across the dataset

fig, axes = plt.subplots(3, n_faces, figsize=(13, 6))

row_labels = ['Original', f'PCA ($k={K}$)', f'NMF ($k={K}$)']

for col, idx in enumerate(face_ids):
    original  = X[idx].reshape(64, 64)
    pca_recon = X_pca_recon[idx].reshape(64, 64)
    nmf_recon = X_nmf_recon[idx].reshape(64, 64)

    for row, (img, label) in enumerate(zip(
        [original, pca_recon, nmf_recon], row_labels
    )):
        axes[row, col].imshow(np.clip(img, 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(label, fontsize=9, rotation=0,
                                      ha='right', labelpad=60)

fig.suptitle(
    'Reconstruction comparison — PCA vs NMF with $k = 16$ components\n'
    'PCA has lower reconstruction error (Eckart–Young); NMF uses only non-negative combinations.',
    fontsize=10
)
plt.tight_layout()
plt.show()

# Per-face reconstruction errors
print(f'{"Face":>5}  {"PCA MSE":>10}  {"NMF MSE":>10}')
print('-' * 30)
for idx in face_ids:
    p_err = np.mean((X[idx] - X_pca_recon[idx])**2)
    n_err = np.mean((X[idx] - X_nmf_recon[idx])**2)
    print(f'{idx:>5}  {p_err:>10.5f}  {n_err:>10.5f}')

In [ ]:
# ── PCA components vs NMF components side by side ────────────────────────────
# Each method found K=16 components. We plot them in two rows so the
# contrast is immediate: PCA components are global and wave-like (and can
# be negative, shown in red/blue); NMF components are non-negative and
# localised to face parts (eyes, nose, forehead, etc.)

fig, axes = plt.subplots(2, K, figsize=(18, 4))

for j in range(K):
    # ── PCA row ──────────────────────────────────────────────────────────────
    comp_pca = pca.components_[j].reshape(64, 64)
    # Diverging colourmap: red = positive weight, blue = negative weight
    axes[0, j].imshow(comp_pca, cmap='RdBu_r',
                      vmin=-np.abs(comp_pca).max(),
                      vmax= np.abs(comp_pca).max())
    axes[0, j].set_title(f'{j+1}', fontsize=7)
    axes[0, j].axis('off')

    # ── NMF row ──────────────────────────────────────────────────────────────
    comp_nmf = nmf.components_[j].reshape(64, 64)
    axes[1, j].imshow(comp_nmf, cmap='gray', vmin=0, vmax=comp_nmf.max())
    axes[1, j].set_title(f'{j+1}', fontsize=7)
    axes[1, j].axis('off')

# Row labels on the left
axes[0, 0].set_ylabel('PCA\n(eigenfaces)', fontsize=10,
                       rotation=0, ha='right', labelpad=70)
axes[1, 0].set_ylabel('NMF\n(parts)',      fontsize=10,
                       rotation=0, ha='right', labelpad=70)

fig.suptitle(
    f'PCA vs NMF components — $k = {K}$\n'
    'PCA: global, wave-like, positive and negative weights (red/blue)\n'
    'NMF: localised face parts, all non-negative (grayscale)',
    fontsize=11
)
plt.tight_layout()
plt.show()

### Method 3: Linear Autoencoder

An autoencoder is a neural network trained to compress its input to a low-dimensional **bottleneck** $z \in \mathbb{R}^k$ and then reconstruct the original from that code. The loss function is the same reconstruction error as PCA and NMF:
$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^n \|x_i - g_\phi(f_\theta(x_i))\|^2.$$

We start with a **linear** autoencoder: both the encoder $f_\theta(x) = Wx$ and decoder $g_\phi(z) = Uz$ are linear maps. 

**Theorem (Baldi & Hornik, 1989):** A linear autoencoder trained to minimise reconstruction error recovers the PCA subspace. The decoder weights span the same space as the top-$k$ principal components.

We verify this empirically: the linear autoencoder should converge to essentially the same reconstruction error as PCA.

In [ ]:
# ── Define a linear autoencoder in PyTorch ────────────────────────────────────
class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # Encoder: x (input_dim) -> z (latent_dim)
        self.encoder = nn.Linear(input_dim, latent_dim, bias=False)
        # Decoder: z (latent_dim) -> x_hat (input_dim)
        self.decoder = nn.Linear(latent_dim, input_dim, bias=False)

    def forward(self, x):
        z     = self.encoder(x)   # compress
        x_hat = self.decoder(z)   # reconstruct
        return x_hat


# ── Prepare data as PyTorch tensors ───────────────────────────────────────────
X_tensor = torch.tensor(X, dtype=torch.float32)
dataset  = TensorDataset(X_tensor)
loader   = DataLoader(dataset, batch_size=40, shuffle=True)

INPUT_DIM  = X.shape[1]   # 4096
LATENT_DIM = K            # same k = 16 as PCA and NMF

linear_ae = LinearAutoencoder(INPUT_DIM, LATENT_DIM)

# ── Train ─────────────────────────────────────────────────────────────────────
optimiser = optim.Adam(linear_ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

N_EPOCHS = 300
losses   = []

linear_ae.train()
for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    for (batch,) in loader:
        optimiser.zero_grad()
        recon = linear_ae(batch)
        loss  = criterion(recon, batch)
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item() * len(batch)
    losses.append(epoch_loss / len(X))
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  MSE = {losses[-1]:.5f}')

linear_ae.eval()
with torch.no_grad():
    X_lae_recon = linear_ae(X_tensor).numpy()
lae_error = np.mean((X - X_lae_recon)**2)

print(f'\nLinear AE MSE : {lae_error:.5f}')
print(f'PCA       MSE : {pca_error:.5f}  <-- should be similar (Baldi–Hornik theorem)')

In [ ]:
# ── Training loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(losses, color='#4878D0', linewidth=1.8)
ax.axhline(pca_error, color='#EE854A', linestyle='--', linewidth=1.5,
           label=f'PCA MSE = {pca_error:.5f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean squared error')
ax.set_title('Linear autoencoder training loss\n'
             'The dashed line is the PCA reconstruction error — the theoretical minimum\n'
             'for any linear method (Baldi & Hornik, 1989)')
ax.legend()
plt.tight_layout()
plt.show()

### Method 4: Nonlinear Autoencoder

Now we add **nonlinear activation functions** (ReLU) between layers. This breaks the Baldi–Hornik equivalence with PCA — the autoencoder can now discover curved, nonlinear structure in the data.

With a small dataset like Olivetti faces (400 images), the improvement over PCA will be modest. On larger datasets (e.g., MNIST with 60,000 images), nonlinear autoencoders substantially outperform PCA. The key point here is not the numbers — it is the **architecture**: the encoder and decoder are nonlinear maps $f_\theta, g_\phi$, and the bottleneck $z$ is a learned nonlinear representation.

### Understanding the Nonlinear Autoencoder Architecture

The network has two parts — an encoder that compresses and a decoder that reconstructs.
The full data flow is:

$$4096 \to 256 \to 256 \to \mathbf{16} \to 256 \to 256 \to 4096$$

where the bold **16** is the bottleneck — this is $z$, the latent code.

---

**Encoder** — compresses from image to latent code:

$$x \in \mathbb{R}^{4096} \xrightarrow[\text{act: ReLU}]{\text{Linear (fully connected)}} \mathbb{R}^{256} \xrightarrow[\text{act: none}]{\text{Linear (fully connected)}} z \in \mathbb{R}^{16}$$

**Decoder** — reconstructs from latent code back to image:

$$z \in \mathbb{R}^{16} \xrightarrow[\text{act: ReLU}]{\text{Linear (fully connected)}} \mathbb{R}^{256} \xrightarrow[\text{act: Sigmoid}]{\text{Linear (fully connected)}} \hat{x} \in \mathbb{R}^{4096}$$

---

**What each layer does**

- `nn.Linear(input_dim, hidden_dim)` is a **fully connected layer**: every input neuron connects to every output neuron with its own learned weight. Mathematically it computes $h = Wx + b$ where $W$ is a matrix of learned weights and $b$ is a bias vector. On its own it is purely linear.

- `nn.ReLU()` applies $\max(0, \cdot)$ elementwise. This is the **nonlinearity** — it is what makes this strictly more powerful than the linear autoencoder. Without it, stacking two linear layers would still be equivalent to a single linear map, and the network would just rediscover PCA.

- `nn.Sigmoid()` at the very end squashes all outputs to $(0, 1)$, matching the pixel value range. This is a practical detail to keep reconstructed pixel values valid.

---

**Parameter count**

| Layer | Calculation | Parameters |
|-------|-------------|------------|
| Encoder: $4096 \to 256$ | $4096 \times 256 + 256$ | ~1,049,000 |
| Encoder: $256 \to 16$ | $256 \times 16 + 16$ | ~4,100 |
| Decoder: $16 \to 256$ | $16 \times 256 + 256$ | ~4,400 |
| Decoder: $256 \to 4096$ | $256 \times 4096 + 4096$ | ~1,053,000 |
| **Total** | | **~2.1 million** |

Training ~2 million parameters on only 400 images means this network is in a regime where
overfitting is a real concern. On a larger dataset such as MNIST (60,000 images) the
nonlinear autoencoder has much more room to learn representations that genuinely
outperform PCA. On Olivetti faces the improvement is modest — but the architecture
is the point, not the numbers.

In [ ]:
# ── Define a nonlinear autoencoder ────────────────────────────────────────────
class NonlinearAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim=256):
        super().__init__()
        # Encoder: input -> hidden -> latent
        # ReLU introduces nonlinearity — the encoder is no longer a simple linear map
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        # Decoder: latent -> hidden -> output
        # Sigmoid at the end keeps reconstructed pixel values in (0, 1)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z     = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


nonlinear_ae = NonlinearAutoencoder(INPUT_DIM, LATENT_DIM, hidden_dim=256)
optimiser_nl = optim.Adam(nonlinear_ae.parameters(), lr=1e-3)

N_EPOCHS_NL = 500
losses_nl   = []

nonlinear_ae.train()
for epoch in range(N_EPOCHS_NL):
    epoch_loss = 0.0
    for (batch,) in loader:
        optimiser_nl.zero_grad()
        recon = nonlinear_ae(batch)
        loss  = criterion(recon, batch)
        loss.backward()
        optimiser_nl.step()
        epoch_loss += loss.item() * len(batch)
    losses_nl.append(epoch_loss / len(X))
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS_NL}  MSE = {losses_nl[-1]:.5f}')

nonlinear_ae.eval()
with torch.no_grad():
    X_nlae_recon = nonlinear_ae(X_tensor).numpy()
nlae_error = np.mean((X - X_nlae_recon)**2)

print(f'\nNonlinear AE MSE : {nlae_error:.5f}')
print(f'Linear AE    MSE : {lae_error:.5f}')
print(f'PCA          MSE : {pca_error:.5f}')

### The Full Comparison

Now we put all four methods side by side: original, PCA, NMF, linear autoencoder, and nonlinear autoencoder. This is the payoff — the same $k = 16$ bottleneck, four different constraints, four different reconstructions.

| Row | Method | Constraint on encoder/decoder | Solution method |
|-----|--------|-------------------------------|-----------------|
| 1 | Original | — | — |
| 2 | PCA | Linear, orthonormal | Closed form (SVD) |
| 3 | NMF | Linear, non-negative | Iterative (local minimum) |
| 4 | Linear autoencoder | Linear neural network | SGD → recovers PCA subspace |
| 5 | Nonlinear autoencoder | Nonlinear neural network (ReLU) | SGD (local minimum) |

In [ ]:
# ── Final comparison: all methods on the same faces ───────────────────────────
methods = [
    ('Original',               X),
    (f'PCA ($k={K}$)',         X_pca_recon),
    (f'NMF ($k={K}$)',         X_nmf_recon),
    (f'Linear AE ($k={K}$)',   X_lae_recon),
    (f'Nonlinear AE ($k={K}$)',X_nlae_recon),
]

n_faces  = 6
face_ids = [0, 10, 20, 30, 40, 50]

fig, axes = plt.subplots(len(methods), n_faces, figsize=(13, 10))

for row, (label, X_recon) in enumerate(methods):
    for col, idx in enumerate(face_ids):
        img = np.clip(X_recon[idx].reshape(64, 64), 0, 1)
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=9, rotation=0,
                             ha='right', labelpad=90)

fig.suptitle(
    'All methods compared — same $k = 16$ bottleneck, same reconstruction objective\n'
    r'Constraint: orthogonal (PCA) | non-negative (NMF) | linear NN (Lin AE) | nonlinear NN (NL AE)',
    fontsize=10
)
plt.tight_layout()
plt.show()

# ── Error summary table ───────────────────────────────────────────────────────
print('\n' + '='*45)
print(f'{"Method":<25} {"MSE":>10} {"vs PCA":>8}')
print('='*45)
for label, err in [
    ('PCA',           pca_error),
    ('NMF',           nmf_error),
    ('Linear AE',     lae_error),
    ('Nonlinear AE',  nlae_error),
]:
    ratio = err / pca_error
    print(f'{label:<25} {err:>10.5f} {ratio:>7.2f}x')
print('='*45)
print('(PCA is the theoretical minimum for linear methods — Eckart–Young)')

---
## Summary

All three methods minimise the same objective — $\|X - \hat{X}\|_F^2$ — but impose different constraints on the encoder and decoder maps.

| Method | Constraint | Solution | Components look like |
|--------|-----------|----------|---------------------|
| PCA | Orthonormal | Closed form (SVD) | Global, wave-like eigenfaces |
| NMF | Non-negative | Iterative (local min) | Localised face parts |
| Linear AE | Linear neural net | SGD → PCA subspace | Same as PCA (Baldi–Hornik) |
| Nonlinear AE | Nonlinear neural net | SGD (local min) | Learned, not visualisable |

The constraint shapes what structure is discovered, not just how well the reconstruction works.

---

*Math 280 | UC Merced | Spring 2026 | S. Sindi*